# BoW sweep analysis

One notebook that cycles through SL, GRPO, and MaxRL sweeps. Artifacts are discovered from the canonical grids in `src/data/bag_of_words.py`; per-group traces aggregate across seeds (mean in the line, number of seeds in the hover, optional min/max error bars via `show_seed_bar=True`). Kinds whose sweep directory is missing are skipped.

In [1]:
import re

import polars as pl
from plotly.colors import qualitative
from plotly.subplots import make_subplots

from src import get_repo_base
from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)


class _AnalysisWithDsCorr(BagOfWordsAnalysisConfig):
    """Adds a `ds_corr` column (dataset corr per study) to the metric dataframe
    so that `xy_plots` can use dataset corr as an axis."""

    def get_metric_dataframe(self) -> pl.DataFrame:
        df = super().get_metric_dataframe()
        lookup = pl.DataFrame(
            {
                "study": list(self.study_corrs.keys()),
                "ds_corr": list(self.study_corrs.values()),
            }
        )
        return df.join(lookup, on="study", how="left")


def _wrap(analysis):
    return _AnalysisWithDsCorr(
        studies=analysis.studies,
        study_seeds=analysis.study_seeds,
        study_corrs=analysis.study_corrs,
    )


ARTIFACTS = get_repo_base() / "artifacts"
KINDS = [
    (
        "SL",
        BagOfWordsAnalysisConfig.from_sl_sweep,
        dict(study_base=ARTIFACTS / "bow-sl-sweep"),
    ),
    (
        "GRPO",
        BagOfWordsAnalysisConfig.from_grpo_sweep,
        dict(study_base=ARTIFACTS / "bow-grpo-sweep"),
    ),
    (
        "MaxRL (subtract-baseline)",
        BagOfWordsAnalysisConfig.from_maxrl_sweep,
        dict(study_base=ARTIFACTS / "bow-maxrl-sweep", subtract_baseline=True),
    ),
    (
        "MaxRL (no-subtract-baseline)",
        BagOfWordsAnalysisConfig.from_maxrl_sweep,
        dict(study_base=ARTIFACTS / "bow-maxrl-sweep", subtract_baseline=False),
    ),
]

epoch_axis = pl.col("epoch").alias("epoch")
ds_corr_axis = pl.col("ds_corr").alias("dataset_corr")
PANELS_PER_EPOCH = [
    (epoch_axis, rsq_expr(split="train", y="target"), None),
    (epoch_axis, rsq_expr(split="train", y="ground_truth"), None),
    (epoch_axis, corr_expr(split="train", y="ground_truth"), None),
]
PANELS_BEST = [
    (ds_corr_axis, corr_expr(split="train", y="target"), True),
    (ds_corr_axis, corr_expr(split="train", y="ground_truth"), True),
]
LOG_X_BEST_PANELS = {0, 1}  # 0-indexed: both dataset_corr panels use log x-axis


def _split_by_rollouts(analysis):
    """Group GRPO/MaxRL studies by the `r=<N>` suffix in their name."""
    groups: dict[int, list[str]] = {}
    for name in analysis.studies:
        m = re.search(r" r=(\d+)", name)
        if m is None:
            return None
        groups.setdefault(int(m.group(1)), []).append(name)
    return dict(sorted(groups.items()))


def _subset(analysis, names):
    return type(analysis)(
        studies={k: analysis.studies[k] for k in names},
        study_seeds={k: analysis.study_seeds[k] for k in names},
        study_corrs={k: analysis.study_corrs[k] for k in names},
    )


def aggregated_by_rollouts(
    analysis, panels, *, title, show_seed_bar=False, log_x_panels=()
):
    """Per-rollouts aggregated (best-epoch) plot. Aggregation:
    mean across seeds per (study, epoch), then argmax epoch per study — same as
    `xy_plots(..., agg=True)`, but one curve per rollout value rather than one
    zigzag trace crossing all rollouts. `log_x_panels` is a set of 0-indexed
    panel positions that should use a log-scale x-axis."""
    groups = _split_by_rollouts(analysis)
    if groups is None:
        fig = analysis.xy_plots(panels, title=title, show_seed_bar=show_seed_bar)
    else:
        fig = make_subplots(rows=1, cols=len(panels))
        palette = qualitative.Plotly
        for i, (rollouts, names) in enumerate(groups.items()):
            sub_fig = _subset(analysis, names).xy_plots(
                panels, show_seed_bar=show_seed_bar
            )
            color = palette[i % len(palette)]
            for j, trace in enumerate(sub_fig.data):
                trace.name = f"r={rollouts}"
                trace.legendgroup = f"r={rollouts}"
                trace.showlegend = (j == 0)
                trace.line.color = color
                trace.marker.color = color
            fig.add_traces(sub_fig.data)
        for col, (x_expr, y_expr, _) in enumerate(panels, start=1):
            fig.update_xaxes(title_text=x_expr.meta.output_name(), row=1, col=col)
            fig.update_yaxes(title_text=y_expr.meta.output_name(), row=1, col=col)
        if title is not None:
            fig.update_layout(title=title)
    for idx in log_x_panels:
        fig.update_xaxes(type="log", row=1, col=idx + 1)
    return fig

## Per-kind plots

For each sweep kind: one per-epoch figure (lines in `epoch`) and one best-epoch summary (one point per group at its seed-averaged optimum).

In [2]:
for label, factory, kwargs in KINDS:
    analysis = factory(**kwargs)
    if analysis is None:
        print(f"[{label}] no artifacts at {kwargs['study_base']} \u2014 skipping")
        continue
    analysis = _wrap(analysis)
    n_runs = sum(len(v) for v in analysis.studies.values())
    max_seeds = max(len(v) for v in analysis.studies.values())
    print(
        f"[{label}] {n_runs} runs across {len(analysis.studies)} groups "
        f"(up to {max_seeds} seeds / group)"
    )
    display(analysis.xy_plots(PANELS_PER_EPOCH, title=f"{label}: per-epoch"))
    display(
        aggregated_by_rollouts(
            analysis,
            PANELS_BEST,
            title=f"{label}: best-epoch",
            log_x_panels=LOG_X_BEST_PANELS,
        )
    )

[SL] 32 runs across 16 groups (up to 2 seeds / group)


[GRPO] 44 runs across 44 groups (up to 1 seeds / group)


[MaxRL (subtract-baseline)] no artifacts at /home/nlyu/Code/maxrl-statistics/artifacts/bow-maxrl-sweep — skipping
[MaxRL (no-subtract-baseline)] no artifacts at /home/nlyu/Code/maxrl-statistics/artifacts/bow-maxrl-sweep — skipping


## Demo: `show_seed_bar=True`

Opt-in min/max-of-seeds error bars overlayed on the mean. Demonstrated on the first kind that has artifacts.

In [3]:
for label, factory, kwargs in KINDS:
    analysis = factory(**kwargs)
    if analysis is None:
        continue
    display(
        analysis.xy_plots(
            PANELS_PER_EPOCH[:2],
            title=f"{label}: per-epoch (seed bars)",
            show_seed_bar=True,
        )
    )
    break